In [ ]:
# ── 셀 0: Kd 사전 추정값 (서열 기반 ΔG, 310K) ─────────────────────────────
# PRODIGY-like empirical formula + CDR 전하상보성 + ESM-2 PLL
kd_estimates = [
    {'id': 'FAP-scFv-1536_H3-C',  'dG': -16.17, 'Kd_nM': 0.004,  'Kd': '4.0 pM',  'note': 'K+E+YYY → E311/D313/R356/F358 ★'},
    {'id': 'FAP-scFv-4766_H3-D',  'dG': -14.70, 'Kd_nM': 0.044,  'Kd': '43.8 pM', 'note': 'K+YYYY'},
    {'id': 'FAP-scFv-6446_H3-B',  'dG': -14.65, 'Kd_nM': 0.048,  'Kd': '48.0 pM', 'note': 'F+K+YYYY'},
    {'id': 'FAP-scFv-13034_H3-E', 'dG': -13.85, 'Kd_nM': 0.174,  'Kd': '174 pM',  'note': '5×Y'},
    {'id': 'FAP-scFv-12534_H3-A', 'dG': -13.79, 'Kd_nM': 0.192,  'Kd': '192 pM',  'note': 'YYY+F ESM-2 best'},
]
print('=== Kd 사전 추정값 (서열 기반, ±2 kcal/mol) ===')
print(f'{"후보":<24} {"ΔG(kcal/mol)":>14} {"Kd":>12}  설명')
print('-'*70)
for k in kd_estimates:
    star = ' ★' if '★' in k['note'] else ''
    print(f"{k['id']:<24} {k['dG']:>+14.2f} {k['Kd']:>12}  {k['note']}{star}")
print('\n⚠ 구조 기반 확인 필요 (ColabFold ipTM + SPR/BLI 실험)')


# FAP scFv Top5 × FAP ECD 복합체 구조 예측
**ColabFold (AlphaFold2-multimer v3)**

- 표적: FAP β-프로펠러 Blade 6-7 (잔기 308–361)
- 후보: Top5 scFv (트라스투주맙 FR, 특허 회피 완료)
- 평가: ipTM ≥ 0.5, Blade 6-7 접촉 (E311/D313/R356/K360/F358)

**실행 방법**: 메뉴 → 런타임 → 모두 실행 (Ctrl+F9)

In [ ]:
# ── 셀 1: GPU 확인 ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'GPU 없음 — 런타임 유형을 T4 GPU로 변경하세요')
print('런타임 변경: 런타임 → 런타임 유형 변경 → T4 GPU')

In [ ]:
# ── 셀 2: ColabFold 설치 (~5분) ────────────────────────────────────────────
import os

if not os.path.exists('/usr/local/lib/python3.10/dist-packages/colabfold'):
    print('ColabFold 설치 중...')
    os.system('pip install -q "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold"')
    os.system('pip install -q --upgrade "jax[cuda11_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html 2>/dev/null || true')
    print('설치 완료')
else:
    print('ColabFold 이미 설치됨')

In [ ]:
# ── 셀 3: 입력 서열 설정 ──────────────────────────────────────────────────
import os, json

os.makedirs('results', exist_ok=True)

# FAP ECD (1Z68 chain A, 367aa) + scFv Top5
# 형식: FAP_ECD:scFv  (ColabFold multimer colon separator)
CANDIDATES = [
    {
        'id': 'FAP-scFv-12534_H3-A',
        'seq': 'GQQSAGSPFPVNFTQKNWLSLAAQRALFQTLQKASSDSGIYMVNQTPQGSDAGVLVYSGVIESGSIRLSWVQHNPYFDVIAHHPQKLAFSTEKSTSSPQAKLNVTPQLEEWRQTLRSHIQFNYGTSTTDATLKPGSQTIEVNLASSDVTPDPETLLPNSNLKNLQSTKYSQDKFQNLSQMDTLSAEYQAHSGKSVVTIDTDHFRLFSSSHQYVLVEHKSATTSFYEFAVGQSSMTQVNMKYTFQLSQNDTRVQMNDNPVISMRSGYFMSATLPKDIDVLPIQKTSALNFKTYNKYVLEFYTPEETFHKAAKMGQINLQSNYQILALDHTVKPSKLDSVFSSALSFIHQAQFDHILSLFNHYEAYTLR:EVQLVESGGGLVQPGGSLRLSCAASGFSITSYYIHWVRQAPGKGLEWVARIISSYSYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARYYGSSGYFAYWGQGTLVTVSSGGGGSGGGGSGGGGSGGGGSDIQMTQSPSSLSASVGDRVTITCRASQSVSTFLSWYQQKPGKAPKLLIYSASSYPSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCRQSYSYPYTFGQGTKVEIK'
    },
    {
        'id': 'FAP-scFv-13034_H3-E',
        'seq': 'GQQSAGSPFPVNFTQKNWLSLAAQRALFQTLQKASSDSGIYMVNQTPQGSDAGVLVYSGVIESGSIRLSWVQHNPYFDVIAHHPQKLAFSTEKSTSSPQAKLNVTPQLEEWRQTLRSHIQFNYGTSTTDATLKPGSQTIEVNLASSDVTPDPETLLPNSNLKNLQSTKYSQDKFQNLSQMDTLSAEYQAHSGKSVVTIDTDHFRLFSSSHQYVLVEHKSATTSFYEFAVGQSSMTQVNMKYTFQLSQNDTRVQMNDNPVISMRSGYFMSATLPKDIDVLPIQKTSALNFKTYNKYVLEFYTPEETFHKAAKMGQINLQSNYQILALDHTVKPSKLDSVFSSALSFIHQAQFDHILSLFNHYEAYTLR:EVQLVESGGGLVQPGGSLRLSCAASGFSITSYYIHWVRQAPGKGLEWVARIISSYSYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARSSYYGYYYYYWGQGTLVTVSSGGGGSGGGGSGGGGSGGGGSDIQMTQSPSSLSASVGDRVTITCRASQSVSTFLSWYQQKPGKAPKLLIYSASSYPSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCRQSYSYPYTFGQGTKVEIK'
    },
    {
        'id': 'FAP-scFv-6446_H3-B',
        'seq': 'GQQSAGSPFPVNFTQKNWLSLAAQRALFQTLQKASSDSGIYMVNQTPQGSDAGVLVYSGVIESGSIRLSWVQHNPYFDVIAHHPQKLAFSTEKSTSSPQAKLNVTPQLEEWRQTLRSHIQFNYGTSTTDATLKPGSQTIEVNLASSDVTPDPETLLPNSNLKNLQSTKYSQDKFQNLSQMDTLSAEYQAHSGKSVVTIDTDHFRLFSSSHQYVLVEHKSATTSFYEFAVGQSSMTQVNMKYTFQLSQNDTRVQMNDNPVISMRSGYFMSATLPKDIDVLPIQKTSALNFKTYNKYVLEFYTPEETFHKAAKMGQINLQSNYQILALDHTVKPSKLDSVFSSALSFIHQAQFDHILSLFNHYEAYTLR:EVQLVESGGGLVQPGGSLRLSCAASGYSITSYYIHWVRQAPGKGLEWVARIISSYSYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARFKGSYYYYYYWGQGTLVTVSSGGGGSGGGGSGGGGSGGGGSDIQMTQSPSSLSASVGDRVTITCRASQSISTYISWYQQKPGKAPKLLIYFASSYPSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQSYSYPFTFGQGTKVEIK'
    },
    {
        'id': 'FAP-scFv-1536_H3-C',
        'seq': 'GQQSAGSPFPVNFTQKNWLSLAAQRALFQTLQKASSDSGIYMVNQTPQGSDAGVLVYSGVIESGSIRLSWVQHNPYFDVIAHHPQKLAFSTEKSTSSPQAKLNVTPQLEEWRQTLRSHIQFNYGTSTTDATLKPGSQTIEVNLASSDVTPDPETLLPNSNLKNLQSTKYSQDKFQNLSQMDTLSAEYQAHSGKSVVTIDTDHFRLFSSSHQYVLVEHKSATTSFYEFAVGQSSMTQVNMKYTFQLSQNDTRVQMNDNPVISMRSGYFMSATLPKDIDVLPIQKTSALNFKTYNKYVLEFYTPEETFHKAAKMGQINLQSNYQILALDHTVKPSKLDSVFSSALSFIHQAQFDHILSLFNHYEAYTLR:EVQLVESGGGLVQPGGSLRLSCAASGYSISSYYIHWVRQAPGKGLEWVARIIFSYSYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCAREYKSSGYYYWGQGTLVTVSSGGGGSGGGGSGGGGSGGGGSDIQMTQSPSSLSASVGDRVTITCRASQSVSTFLSWYQQKPGKAPKLLIYRASSYPSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQSYSYPFTFGQGTKVEIK'
    },
    {
        'id': 'FAP-scFv-4766_H3-D',
        'seq': 'GQQSAGSPFPVNFTQKNWLSLAAQRALFQTLQKASSDSGIYMVNQTPQGSDAGVLVYSGVIESGSIRLSWVQHNPYFDVIAHHPQKLAFSTEKSTSSPQAKLNVTPQLEEWRQTLRSHIQFNYGTSTTDATLKPGSQTIEVNLASSDVTPDPETLLPNSNLKNLQSTKYSQDKFQNLSQMDTLSAEYQAHSGKSVVTIDTDHFRLFSSSHQYVLVEHKSATTSFYEFAVGQSSMTQVNMKYTFQLSQNDTRVQMNDNPVISMRSGYFMSATLPKDIDVLPIQKTSALNFKTYNKYVLEFYTPEETFHKAAKMGQINLQSNYQILALDHTVKPSKLDSVFSSALSFIHQAQFDHILSLFNHYEAYTLR:EVQLVESGGGLVQPGGSLRLSCAASGYTISSYYIHWVRQAPGKGLEWVARIIFSYSYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARKYGSYYYGYYYWGQGTLVTVSSGGGGSGGGGSGGGGSGGGGSDIQMTQSPSSLSASVGDRVTITCRASQSISTYLSWYQQKPGKAPKLLIYYASSRPSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQSYSYPFTFGQGTKVEIK'
    },
]

# FASTA 파일 생성
with open('fap_top5_complex.fasta', 'w') as f:
    for c in CANDIDATES:
        f.write(f">{c['id']}\n{c['seq']}\n")

print(f'입력 파일 생성: fap_top5_complex.fasta ({len(CANDIDATES)}개 복합체)')
print('FAP ECD: 367aa, scFv: ~246aa, 합계: ~613aa/쌍')

In [ ]:
# ── 셀 4: ColabFold 실행 (~20-30분/후보, T4 기준) ──────────────────────────
import subprocess, time

NUM_RECYCLES  = 3    # 빠른 스크리닝: 3, 고품질: 12-20
NUM_MODELS    = 5    # AlphaFold2 모델 앙상블 수
MODEL_TYPE    = 'alphafold2_multimer_v3'

print(f'ColabFold 실행 시작')
print(f'  모델: {MODEL_TYPE}, recycles={NUM_RECYCLES}, models={NUM_MODELS}')
print(f'  예상 시간: ~{len(CANDIDATES) * 25}분 (T4 기준)\n')

t0 = time.time()
cmd = [
    'colabfold_batch',
    'fap_top5_complex.fasta',
    'results/',
    '--model-type', MODEL_TYPE,
    '--num-recycle', str(NUM_RECYCLES),
    '--num-models', str(NUM_MODELS),
    '--rank', 'ipTM',
    '--use-gpu-relax',
]

result = subprocess.run(cmd, capture_output=False, text=True)
elapsed = time.time() - t0
print(f'\n완료: {elapsed/60:.1f}분')

In [ ]:
# ── 셀 5: 결과 파싱 (ipTM / pTM / pLDDT) ─────────────────────────────────
import glob, json, os
import numpy as np

BLADE67 = set(range(308, 362))   # FAP Blade 6-7 잔기
KEY_RES = {311:'E311', 313:'D313', 356:'R356', 358:'F358', 360:'K360'}
CUTOFF  = 4.5  # Å 접촉 거리

def parse_scores(scores_file):
    with open(scores_file) as f:
        d = json.load(f)
    plddt = d.get('plddt', [])
    return {
        'iptm': d.get('iptm'),
        'ptm':  d.get('ptm'),
        'plddt_mean': round(sum(plddt)/len(plddt), 2) if plddt else None,
        'ranking_confidence': d.get('ranking_confidence'),
    }

def parse_ca_pdb(pdb_path):
    """chain A/B Cα 좌표 파싱."""
    ca = {'A': {}, 'B': {}}
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith('ATOM'): continue
            if line[12:16].strip() != 'CA': continue
            chain = line[21]
            if chain not in ca: continue
            resnum = int(line[22:26])
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            ca[chain][resnum] = np.array([x,y,z])
    return ca

def contact_analysis(pdb_path):
    if not pdb_path or not os.path.exists(pdb_path):
        return {'n_blade67': 0, 'key_contacts': {}}
    ca = parse_ca_pdb(pdb_path)
    contacts = []
    for fap_res, fap_coord in ca['A'].items():
        if fap_res not in BLADE67: continue
        for scfv_res, scfv_coord in ca['B'].items():
            d = float(np.linalg.norm(fap_coord - scfv_coord))
            if d <= CUTOFF:
                contacts.append({'fap': fap_res, 'scfv': scfv_res, 'dist': round(d,2),
                                  'label': KEY_RES.get(fap_res, f'FAP{fap_res}')})
    key_c = {}
    for res, label in KEY_RES.items():
        hits = [c for c in contacts if c['fap']==res]
        key_c[label] = sorted(hits, key=lambda x: x['dist'])[:3]
    return {'n_blade67': len(contacts), 'contacts': contacts[:10], 'key_contacts': key_c}

# 결과 수집
summary = []
for cand in CANDIDATES:
    cid = cand['id']
    # best model = rank_001
    score_files = sorted(glob.glob(f'results/*{cid.split("_")[0]}*scores_rank_001*.json'))
    if not score_files:
        score_files = sorted(glob.glob(f'results/*scores_rank_001*.json'))
    
    if not score_files:
        print(f'{cid}: 결과 없음')
        continue
    
    sc = parse_scores(score_files[0])
    
    # PDB
    pdb_f = score_files[0].replace('scores_rank_001','relaxed_rank_001').replace('.json','.pdb')
    if not os.path.exists(pdb_f):
        pdb_f = score_files[0].replace('scores_rank_001','unrelaxed_rank_001').replace('.json','.pdb')
    contacts = contact_analysis(pdb_f)
    
    row = {**sc, 'id': cid, **contacts, 'pass_iptm': (sc.get('iptm') or 0) >= 0.5}
    summary.append(row)
    
    iptm_s = f"{sc['iptm']:.3f}" if sc.get('iptm') else 'N/A'
    ptm_s  = f"{sc['ptm']:.3f}"  if sc.get('ptm')  else 'N/A'
    mark   = '✅' if row['pass_iptm'] else '❌'
    print(f"{mark} {cid}")
    print(f"   ipTM={iptm_s}  pTM={ptm_s}  pLDDT={sc.get('plddt_mean')}  Blade6-7접촉={contacts['n_blade67']}")
    for label, hits in contacts['key_contacts'].items():
        if hits:
            print(f"   {label} → scFv잔기{hits[0]['scfv']} {hits[0]['dist']}Å")
    print()

In [ ]:
# ── 셀 6: 순위표 및 최종 선정 ────────────────────────────────────────────
import pandas as pd

if summary:
    df = pd.DataFrame(summary)[['id','iptm','ptm','plddt_mean','n_blade67','pass_iptm']]
    df = df.sort_values('iptm', ascending=False).reset_index(drop=True)
    df.index += 1
    print('=== ColabFold 결과 순위 ===')
    print(df.to_string())
    print(f"\nipTM ≥ 0.5 통과: {df['pass_iptm'].sum()}개")
    print(f"최고 ipTM: {df.iloc[0]['id']} ({df.iloc[0]['iptm']:.3f})")
    print("\n→ MD 후보:", ', '.join(df[df['pass_iptm']]['id'].tolist()[:3]))

    # JSON 저장
    with open('results/complex_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print('\n[저장] results/complex_summary.json')

In [ ]:
# ── 셀 7: Google Drive 저장 + ZIP 다운로드 ────────────────────────────────
import shutil, os
from google.colab import drive, files

# Google Drive 마운트
drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/FAP_ColabFold_results'
os.makedirs(drive_path, exist_ok=True)

# 결과 복사
shutil.copytree('results', drive_path + '/results', dirs_exist_ok=True)
print(f'Google Drive 저장: {drive_path}')

# ZIP 다운로드
shutil.make_archive('FAP_ColabFold_results', 'zip', 'results')
print('ZIP 다운로드 시작...')
files.download('FAP_ColabFold_results.zip')
print('완료!')
print('\n다운받은 ZIP을 Antibody/fap_design/colabfold/results/ 에 압축 해제하세요')

In [ ]:
# ── 셀 9: 시각화 (ipTM + Blade 6-7 + Kd) ────────────────────────────────
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    kd_data = {
        'FAP-scFv-1536':  0.004,
        'FAP-scFv-12534': 0.192,
        'FAP-scFv-13034': 0.174,
        'FAP-scFv-6446':  0.048,
        'FAP-scFv-4766':  0.044,
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if summary:
        ids   = [s['id'].split('_')[0] for s in summary]
        iptms = [s.get('iptm') or 0 for s in summary]
        colors = ['#0a7a52' if v >= 0.5 else '#b52020' for v in iptms]
        axes[0].barh(ids, iptms, color=colors)
        axes[0].axvline(0.5, color='gray', linestyle='--')
        axes[0].set_xlabel('ipTM')
        axes[0].set_title('ipTM (복합체 신뢰도)')

        contacts = [s.get('n_blade67') or 0 for s in summary]
        axes[1].barh(ids, contacts, color='#0d8b8b')
        axes[1].set_xlabel('접촉 수')
        axes[1].set_title('Blade 6-7 접촉 (≤4.5Å)')

    # Kd bar (log scale)
    import math
    kd_ids = list(kd_data.keys())
    kd_vals = [-math.log10(v * 1e-9) for v in kd_data.values()]  # pKd
    bar_colors = ['#e65c00' if k=='FAP-scFv-1536' else '#2563a8' for k in kd_ids]
    axes[2].barh(kd_ids, kd_vals, color=bar_colors)
    axes[2].axvline(8.0, color='gray', linestyle='--', label='Kd=10 nM')
    axes[2].set_xlabel('pKd = -log₁₀[Kd(M)]')
    axes[2].set_title('예상 Kd (서열 기반)')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig('results/colabfold_summary.png', dpi=150)
    plt.show()
    print('[저장] results/colabfold_summary.png')
except Exception as e:
    print(f'시각화 건너뜀: {e}')
